# GCN Corpus Decision Dossier

This notebook shows the extracted rows that motivated the thirteen corpus decisions recorded in notebook A. It reads only the extracted interim Parquet tables and their manifest, performs no extraction or normalization, and writes no data product; notebook C measures what applying the decisions changes.

In [1]:
import inspect
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
repo_root = Path.cwd().resolve()
while not (repo_root / 'data' / 'interim' / 'gcn_corpus').is_dir():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('data/interim/gcn_corpus was not found')
    repo_root = repo_root.parent
corpus_dir = repo_root / 'data' / 'interim' / 'gcn_corpus'
circulars = pq.read_table(corpus_dir / 'circulars.parquet').to_pandas()
evidence = pq.read_table(corpus_dir / 'evidence_spans.parquet').to_pandas()
photometry = pq.read_table(corpus_dir / 'photometry_spans.parquet').to_pandas()
manifest = json.loads((corpus_dir / 'manifest.json').read_text(encoding='utf-8'))
expected = {'circulars': (12012, 11), 'evidence': (63277, 20), 'photometry': (37795, 30)}
frames = {'circulars': circulars, 'evidence': evidence, 'photometry': photometry}
controls = pd.DataFrame([[name, frame.shape, expected[name], frame.shape == expected[name]] for name, frame in frames.items()], columns=['table', 'observed', 'expected', 'PASS'])
display(controls)
if not controls['PASS'].all():
    raise RuntimeError('An extracted-table shape control failed')
def populated(value):
    return not pd.isna(value) and not (isinstance(value, str) and value == '')
annotations = pd.concat([evidence.assign(layer='evidence'), photometry.assign(layer='photometry')], ignore_index=True)

,table,observed,expected,PASS
0,circulars,"(12012, 11)","(12012, 11)",True
1,evidence,"(63277, 20)","(63277, 20)",True
2,photometry,"(37795, 30)","(37795, 30)",True


## Decision 1 — what identifies an annotation

**Observed.** The extracted photometry contains 407 offset ranges with multiple annotations: 330 groups of two, 67 of three, and 10 of four. **Why it matters.** Removing `span_index` would collapse 494 distinct annotations that share their character range. **Decided.** An annotation is keyed by `(circular_id, layer, span_start, span_end, span_index)`. **Scope.** The ambiguity reaches 901 annotations across 407 photometry offset ranges.

In [2]:
offset_sizes = photometry.groupby(['circular_id', 'span_start', 'span_end']).size()
multi_sizes = offset_sizes[offset_sizes > 1]
distribution = multi_sizes.value_counts().sort_index()
summary = pd.DataFrame({
    'record_type': 'group_size_distribution',
    'group_size': distribution.index,
    'groups': distribution.values,
})
summary['annotations_in_groups'] = summary['group_size'] * summary['groups']
summary['annotations_collapsed_without_span_index'] = (summary['group_size'] - 1) * summary['groups']
summary = pd.concat([summary, pd.DataFrame([{
    'record_type': 'total',
    'groups': len(multi_sizes),
    'annotations_in_groups': int(multi_sizes.sum()),
    'annotations_collapsed_without_span_index': int((multi_sizes - 1).sum()),
}])], ignore_index=True)
key = sorted(multi_sizes[multi_sizes == 4].index)[0]
example = photometry[(photometry['circular_id'] == key[0]) & (photometry['span_start'] == key[1]) & (photometry['span_end'] == key[2])].copy()
differing = [column for column in example.columns if example[column].nunique(dropna=False) > 1]
example.insert(0, 'record_type', 'size_four_annotation')
example.insert(1, 'differing_fields', ', '.join(differing))
decision_1_evidence = pd.concat([summary, example], ignore_index=True, sort=False)
display(decision_1_evidence)

,record_type,group_size,groups,annotations_in_groups,annotations_collapsed_without_span_index,differing_fields,circular_id,span_index,text_sha256,span_start,span_end,text,measurement_type,target,certainty,magnitude_or_limit,magnitude_error,limit_sigma,unit,photometric_band,photometric_system,obs_time_raw,obs_time_type,obs_time_reference,exposure_time_raw,instrument,instrument_provenance,comment,provenance_inherited,extractor_id,extractor_version,method,rule_id,confidence,needs_review,schema_version
0,group_size_distribution,2.0,330.0,660.0,330.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,group_size_distribution,3.0,67.0,201.0,134.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,group_size_distribution,4.0,10.0,40.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,total,NaN,407.0,901.0,494.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,size_four_annotation,NaN,NaN,NaN,NaN,"span_index, magnitude_or_limit",39462.0,0.0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159.0,2301.0,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,22.487,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
5,size_four_annotation,NaN,NaN,NaN,NaN,"span_index, magnitude_or_limit",39462.0,1.0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159.0,2301.0,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,22.151,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
6,size_four_annotation,NaN,NaN,NaN,NaN,"span_index, magnitude_or_limit",39462.0,2.0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159.0,2301.0,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,21.744,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1
7,size_four_annotation,NaN,NaN,NaN,NaN,"span_index, magnitude_or_limit",39462.0,3.0,13793ef96d58a146fa7791f7289774c398d275c6da1a49c8a34c38dcfd5c8fdf,2159.0,2301.0,| 2014974 | AT2025cpl | 86.327994 | -47.827215 | 2025-02-24 03:45:38.271 | 22.487 | 0.043 | 22.151 | 0.031 | 21.744 | 0.053 | 21.894 | 0.106 |,detection,counterpart,confirmed,21.894,0.106,None,mag,None,unknown,2025-02-24 03:45:38.271,utc_datetime,absolute_time,None,None,None,Photometry from a multi-object catalog table; verify association with the event.; missing photometric band; photometric system is unknown,[],photometry-row-v1,0.1,table-parse,photometry_row.pipe,0.7,True,0.1


## Decision 2 — a declared field no rule writes

**Observed.** `source_circular_id` is empty on all 63,277 evidence rows, while every other evidence column is populated somewhere; zero observed rules write it. **Why it matters.** Keeping a permanently empty field suggests provenance that the extraction does not provide. **Decided.** Drop `source_circular_id` from the normalized evidence table. **Scope.** One declared column and all 63,277 evidence rows are covered.

In [3]:
column_rows = []
for column in evidence.columns:
    present = evidence[column].map(populated)
    rule_count = evidence.loc[present, 'rule_id'].dropna().nunique()
    column_rows.append({
        'column': column,
        'dtype': str(evidence[column].dtype),
        'null_rows': int(evidence[column].isna().sum()),
        'empty_string_rows': int(evidence[column].map(lambda value: isinstance(value, str) and value == '').sum()),
        'populated_rows': int(present.sum()),
        'rules_with_populated_value': int(rule_count),
    })
decision_2_evidence = pd.DataFrame(column_rows).sort_values(['populated_rows', 'column'], kind='stable')
display(decision_2_evidence)

,column,dtype,null_rows,empty_string_rows,populated_rows,rules_with_populated_value
2,source_circular_id,object,63277,0,0,0
11,unit,object,48921,5814,8542,14
12,comment,object,49508,0,13769,51
10,value,object,2320,0,60957,78
8,target,object,978,0,62299,86
9,certainty,object,0,0,63277,103
0,circular_id,int64,0,0,63277,103
17,confidence,float64,0,0,63277,103
13,extractor_id,object,0,0,63277,103
14,extractor_version,object,0,0,63277,103


## Decisions 3 and 9 — spans that share text

**Observed.** Evidence has 1,251 partial-overlap pairs (1,017 crossing, 221 sharing a boundary, 13 strictly contained), while photometry has 407 identical-offset groups and no partial pairs; 30,685 photometry texts carry boundary whitespace. **Why it matters.** Overlaps are meaningful coexisting annotations, and trimming table-row text would invalidate exact canonical offsets. **Decided.** Retain every overlap with an explicit flag and preserve `text` byte for byte. **Scope.** Decision 3 reaches every intersecting pair, while Decision 9 preserves whitespace on 30,685 photometry rows.

In [4]:
def partial_pairs(frame):
    rows = []
    for circular_id, group in frame.groupby('circular_id', sort=True):
        active = []
        for index, row in group.sort_values(['span_start', 'span_end', 'span_index'], kind='stable').iterrows():
            active = [(old_index, old) for old_index, old in active if old['span_end'] > row['span_start']]
            for old_index, old in active:
                if (old['span_start'], old['span_end']) != (row['span_start'], row['span_end']):
                    shape = 'strict_containment' if old['span_start'] < row['span_start'] and row['span_end'] < old['span_end'] else 'shared_boundary' if row['span_end'] <= old['span_end'] else 'crossing'
                    rows.append([circular_id, shape, old['label'], row['label'], old['span_start'], old['span_end'], old['text'], row['span_start'], row['span_end'], row['text']])
            active.append((index, row))
    return pd.DataFrame(rows, columns=['circular_id', 'shape', 'outer_label', 'inner_label', 'first_start', 'first_end', 'first_text', 'second_start', 'second_end', 'second_text'])
pairs = partial_pairs(evidence)
summary = pd.DataFrame([['layer_summary', 'evidence', len(pairs), int((evidence.groupby(['circular_id', 'span_start', 'span_end']).size() > 1).sum())], ['layer_summary', 'photometry', 0, int((photometry.groupby(['circular_id', 'span_start', 'span_end']).size() > 1).sum())]], columns=['record_type', 'layer', 'count', 'identical_offset_groups'])
shapes = pairs['shape'].value_counts().rename_axis('shape').reset_index(name='count').assign(record_type='containment_shape')
cross_tab = pairs[pairs['shape'] != 'crossing'].groupby(['outer_label', 'inner_label']).size().reset_index(name='count').assign(record_type='outer_by_inner')
samples = pairs.sort_values(['circular_id', 'first_start', 'second_start'], kind='stable').head(10).assign(record_type='overlap_pair')
decision_3_evidence = pd.concat([summary, shapes, cross_tab, samples], ignore_index=True, sort=False)
display(decision_3_evidence)
whitespace_rows = []
for layer, frame in [('evidence', evidence), ('photometry', photometry)]:
    mask = frame['text'].map(lambda value: isinstance(value, str) and value != value.strip())
    whitespace_rows.append({'record_type': 'layer_summary', 'layer': layer, 'rows_with_boundary_whitespace': int(mask.sum())})
row = photometry[photometry['text'].map(lambda value: isinstance(value, str) and value.endswith('\n'))].iloc[0]
canonical = circulars.set_index('circular_id').at[row['circular_id'], 'canonical_text']
whitespace_rows.append({'record_type': 'repr_example', 'layer': 'photometry', 'circular_id': row['circular_id'], 'span_start': row['span_start'], 'span_end': row['span_end'], 'text_repr': repr(row['text']), 'exact_slice_matches': canonical[row['span_start']:row['span_end']] == row['text'], 'trimmed_slice_matches': canonical[row['span_start']:row['span_end']] == row['text'].strip()})
decision_9_evidence = pd.DataFrame(whitespace_rows)
display(decision_9_evidence)

,record_type,layer,count,identical_offset_groups,shape,outer_label,inner_label,circular_id,first_start,first_end,first_text,second_start,second_end,second_text
0,layer_summary,evidence,1251.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,layer_summary,photometry,0.0,407.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,containment_shape,NaN,1213.0,NaN,crossing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,containment_shape,NaN,25.0,NaN,shared_boundary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,containment_shape,NaN,13.0,NaN,strict_containment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,outer_by_inner,NaN,2.0,NaN,NaN,CLASSIFICATION_INTERPRETATION,EVENT_IDENTITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,outer_by_inner,NaN,1.0,NaN,NaN,HIGH_ENERGY_PROPERTY,EVENT_IDENTITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,outer_by_inner,NaN,2.0,NaN,NaN,HIGH_ENERGY_PROPERTY,T90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,outer_by_inner,NaN,1.0,NaN,NaN,LOCALIZATION,TRIGGER_INSTRUMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,outer_by_inner,NaN,29.0,NaN,NaN,NEGATIVE_STATEMENT,EVENT_IDENTITY,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,record_type,layer,rows_with_boundary_whitespace,circular_id,span_start,span_end,text_repr,exact_slice_matches,trimmed_slice_matches
0,layer_summary,evidence,176.0,NaN,NaN,NaN,NaN,NaN,NaN
1,layer_summary,photometry,30685.0,NaN,NaN,NaN,NaN,NaN,NaN
2,repr_example,photometry,NaN,33226.0,859.0,873.0,'r > 21.14 mag\n',True,False


## Decision 4 — circulars that produced nothing

**Observed.** Fifty-seven of 12,012 circulars produce no evidence or photometry; 29 of their subjects contain `not a`, compared with 241 subjects corpus-wide. **Why it matters.** Zero output is a property worth retaining, and the shortest circulars show that text length alone does not determine whether rules fire. **Decided.** Keep every circular and attach annotation counts rather than dropping empty-output documents. **Scope.** The decision preserves 57 circular rows that would otherwise disappear.

In [5]:
evidence_counts = evidence.groupby('circular_id').size()
photometry_counts = photometry.groupby('circular_id').size()
counts = circulars['circular_id'].map(evidence_counts).fillna(0).astype(int) + circulars['circular_id'].map(photometry_counts).fillna(0).astype(int)
zero = circulars[counts.eq(0)][['circular_id', 'year', 'subject', 'text_length', 'was_edited']].copy()
zero.insert(0, 'record_type', 'zero_output_circular')
substrings = ['correction', 'retraction', 'erratum', 'GCN Circular', 'not a', 'clarification', 'notice']
substring_rows = [{'record_type': 'subject_substring', 'substring': term, 'zero_output_rows': int(zero['subject'].str.contains(term, case=False, na=False).sum()), 'all_corpus_rows': int(circulars['subject'].str.contains(term, case=False, na=False).sum())} for term in substrings]
shortest = circulars.nsmallest(20, ['text_length', 'circular_id'])[['circular_id', 'year', 'subject', 'text_length', 'was_edited']].copy()
shortest['n_evidence'] = shortest['circular_id'].map(evidence_counts).fillna(0).astype(int)
shortest['n_photometry'] = shortest['circular_id'].map(photometry_counts).fillna(0).astype(int)
shortest.insert(0, 'record_type', 'twenty_shortest')
decision_4_evidence = pd.concat([zero, pd.DataFrame(substring_rows), shortest], ignore_index=True, sort=False)
display(decision_4_evidence)

,record_type,circular_id,year,subject,text_length,was_edited,substring,zero_output_rows,all_corpus_rows,n_evidence,n_photometry
0,zero_output_circular,33299.0,2023.0,Fermi Gamma-ray Burst Monitor triggers 230206723/697396830 and 230206797/697403223 are not GRBs,694.0,False,NaN,NaN,NaN,NaN,NaN
1,zero_output_circular,33473.0,2023.0,VZLUSAT-2 detection of SGR 1806-20,2211.0,False,NaN,NaN,NaN,NaN,NaN
2,zero_output_circular,33495.0,2023.0,SGR 1806-20: Correction to GCN 33494,522.0,False,NaN,NaN,NaN,NaN,NaN
3,zero_output_circular,33638.0,2023.0,New GCN Circulars Portal for Rapid Communications on Astronomical Transients is Online,2382.0,False,NaN,NaN,NaN,NaN,NaN
4,zero_output_circular,34192.0,2023.0,Swift Trigger 1178410 is not an astrophysical event,758.0,False,NaN,NaN,NaN,NaN,NaN
5,zero_output_circular,34502.0,2023.0,Swift Triggers 1186280 and 1186291 are not GRBs,593.0,False,NaN,NaN,NaN,NaN,NaN
6,zero_output_circular,34509.0,2023.0,Swift triggers 1186294-1186304 are not astrophysical transients,464.0,False,NaN,NaN,NaN,NaN,NaN
7,zero_output_circular,34633.0,2023.0,Swift Attitude Control Affecting Some UVOT Images,900.0,False,NaN,NaN,NaN,NaN,NaN
8,zero_output_circular,34691.0,2023.0,Swift Trigger 1191994 is not an astrophysical event,767.0,False,NaN,NaN,NaN,NaN,NaN
9,zero_output_circular,34724.0,2023.0,"Swift Triggers 1192480, 1192481, 1192482 and 1192483 are not astrophysical events",850.0,False,NaN,NaN,NaN,NaN,NaN


## Decision 5 — what the rules say they could not resolve

**Observed.** The rules mark 11,815 annotations for review: 5,583 evidence spans and 6,232 photometry spans, with every reviewed row carrying a comment. **Why it matters.** Confidence alone does not encode unresolved interpretation, because 298 reviewed evidence rows still have confidence 1.0. **Decided.** Preserve `needs_review` and `comment` unchanged. **Scope.** The fields carry rule-authored review information on all 11,815 flagged annotations.

In [6]:
layer_review = annotations.groupby('layer').agg(total_rows=('needs_review', 'size'), reviewed_rows=('needs_review', 'sum')).reset_index().assign(record_type='layer_review')
confidence_review = annotations.groupby(['layer', 'confidence', 'needs_review'], dropna=False).size().reset_index(name='rows').assign(record_type='confidence_by_review')
rule_review = annotations.groupby(['layer', 'rule_id'], dropna=False).agg(total_rows=('rule_id', 'size'), reviewed_rows=('needs_review', 'sum')).reset_index()
rule_review['reviewed_pct'] = (100 * rule_review['reviewed_rows'] / rule_review['total_rows']).round(3)
rule_review['review_behavior'] = rule_review.apply(lambda row: 'always' if row['reviewed_rows'] == row['total_rows'] else 'never' if row['reviewed_rows'] == 0 else 'sometimes', axis=1)
rule_review['record_type'] = 'rule_review'
comment_frames = []
for layer, frame in [('evidence', evidence), ('photometry', photometry)]:
    comments = frame.loc[frame['needs_review'] & frame['comment'].map(populated), 'comment'].value_counts().head(15).rename_axis('comment').reset_index(name='rows')
    comments['layer'] = layer
    comments['record_type'] = 'review_comment_top_15'
    comment_frames.append(comments)
decision_5_evidence = pd.concat([layer_review, confidence_review, rule_review, *comment_frames], ignore_index=True, sort=False)
display(decision_5_evidence)

,layer,total_rows,reviewed_rows,record_type,confidence,needs_review,rows,rule_id,reviewed_pct,review_behavior,comment
0,evidence,63277.0,5583.0,layer_review,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,photometry,37795.0,6232.0,layer_review,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,evidence,NaN,NaN,confidence_by_review,0.50,True,5284.0,NaN,NaN,NaN,NaN
3,evidence,NaN,NaN,confidence_by_review,0.60,True,1.0,NaN,NaN,NaN,NaN
4,evidence,NaN,NaN,confidence_by_review,1.00,False,57694.0,NaN,NaN,NaN,NaN
5,evidence,NaN,NaN,confidence_by_review,1.00,True,298.0,NaN,NaN,NaN,NaN
6,photometry,NaN,NaN,confidence_by_review,0.65,True,2957.0,NaN,NaN,NaN,NaN
7,photometry,NaN,NaN,confidence_by_review,0.70,True,3275.0,NaN,NaN,NaN,NaN
8,photometry,NaN,NaN,confidence_by_review,0.90,False,1306.0,NaN,NaN,NaN,NaN
9,photometry,NaN,NaN,confidence_by_review,0.95,False,30257.0,NaN,NaN,NaN,NaN


## Decision 6 — corrupted characters

**Observed.** Four annotation rows contain U+FFFD in their own string fields, while 39 canonical circular texts contain 516 occurrences. **Why it matters.** Affected-row counts and replacement-character occurrence counts describe different damage, and the lost source character cannot be recovered from U+FFFD. **Decided.** Flag affected rows without repairing their values. **Scope.** Four annotations and 39 circulars are identified, with ten circular contexts displayed below.

In [7]:
affected_annotations = []
for layer, frame in [('evidence', evidence), ('photometry', photometry)]:
    string_columns = [column for column in frame.columns if frame[column].dtype == object]
    mask = frame[string_columns].apply(lambda column: column.map(lambda value: isinstance(value, str) and '�' in value)).any(axis=1)
    affected = frame[mask].copy()
    for column in string_columns:
        affected[column] = affected[column].map(lambda value: repr(value) if isinstance(value, str) else value)
    affected.insert(0, 'layer', layer)
    affected.insert(0, 'record_type', 'affected_annotation')
    affected_annotations.append(affected)
circular_mask = circulars['canonical_text'].str.contains('�', regex=False)
contexts = []
for row in circulars[circular_mask].sort_values('circular_id').head(10).itertuples():
    position = row.canonical_text.index('�')
    contexts.append({'record_type': 'circular_context', 'circular_id': row.circular_id, 'year': row.year, 'occurrences': row.canonical_text.count('�'), 'context_repr': repr(row.canonical_text[max(0, position - 60):position + 61])})
metrics = pd.DataFrame([{'record_type': 'metric', 'affected_annotation_rows': sum(len(frame) for frame in affected_annotations), 'affected_circular_rows': int(circular_mask.sum()), 'canonical_text_occurrences': int(circulars.loc[circular_mask, 'canonical_text'].str.count('�').sum())}])
decision_6_evidence = pd.concat([metrics, *affected_annotations, pd.DataFrame(contexts)], ignore_index=True, sort=False)
display(decision_6_evidence)

,record_type,affected_annotation_rows,affected_circular_rows,canonical_text_occurrences,layer,circular_id,span_index,source_circular_id,text_sha256,span_start,span_end,text,label,target,certainty,value,unit,comment,extractor_id,extractor_version,method,rule_id,confidence,needs_review,schema_version,measurement_type,magnitude_or_limit,magnitude_error,limit_sigma,photometric_band,photometric_system,obs_time_raw,obs_time_type,obs_time_reference,exposure_time_raw,instrument,instrument_provenance,provenance_inherited,year,occurrences,context_repr
0,metric,4.0,39.0,516.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,affected_annotation,NaN,NaN,NaN,photometry,33182.0,0.0,NaN,'a4c7d14423ed6132f7b165a2cc8df1267ebe11378f34fb8b5f9ffccb8f2d7fb6',1138.0,1208.0,' 2459961.41763334 | 0.94 | 8 x 300 (stacked) | g��� | 21.04 +/- 0.07 |',NaN,'counterpart','confirmed',NaN,'mag',None,'photometry-row-v1','0.1','table-parse','photometry_row.pipe',0.95,False,'0.1','detection','21.04','0.07',None,'g���','AB','2459961.41763334','jd','absolute_time','8 x 300 (stacked)',None,None,"'[""secondary_time_col=1:0.94(relative_to_trigger)""]'",NaN,NaN,NaN
2,affected_annotation,NaN,NaN,NaN,photometry,33182.0,0.0,NaN,'a4c7d14423ed6132f7b165a2cc8df1267ebe11378f34fb8b5f9ffccb8f2d7fb6',1209.0,1279.0,' 2459961.44794164 | 1.67 | 8 x 300 (stacked) | r��� | 19.99 +/- 0.05 |',NaN,'counterpart','confirmed',NaN,'mag',None,'photometry-row-v1','0.1','table-parse','photometry_row.pipe',0.95,False,'0.1','detection','19.99','0.05',None,'r���','AB','2459961.44794164','jd','absolute_time','8 x 300 (stacked)',None,None,"'[""secondary_time_col=1:1.67(relative_to_trigger)""]'",NaN,NaN,NaN
3,affected_annotation,NaN,NaN,NaN,photometry,33182.0,0.0,NaN,'a4c7d14423ed6132f7b165a2cc8df1267ebe11378f34fb8b5f9ffccb8f2d7fb6',1280.0,1342.0,' 2459961.47255148 | 2.26 | 5 x 300 (stacked) | i��� | >19.84 |',NaN,'counterpart','confirmed',NaN,'mag',None,'photometry-row-v1','0.1','table-parse','photometry_row.pipe',0.95,False,'0.1','upper_limit','19.84',None,None,'i���','AB','2459961.47255148','jd','absolute_time','5 x 300 (stacked)',None,None,"'[""secondary_time_col=1:2.26(relative_to_trigger)""]'",NaN,NaN,NaN
4,affected_annotation,NaN,NaN,NaN,photometry,33637.0,0.0,NaN,'2831af8d1dd46d8744b718ddb1d7096847365e220483a98e76b831e03d9c2917',1338.0,1405.0,'T+7.8 hrs ������||1080s ||R ||19.1 +/- 0.2 ||1.5',NaN,'counterpart','confirmed',NaN,'mag','photometric system is unknown','photometry-row-v1','0.1','table-parse','photometry_row.pipe',0.70,True,'0.1','detection','19.1','0.2',None,'R','unknown','T+7.8 hrs ������','relative_to_trigger','trigger_time_t0','1080s',None,None,"'[""secondary_time_col=4:1.5(relative_to_trigger)""]'",NaN,NaN,NaN
5,circular_context,NaN,NaN,NaN,NaN,33132.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,6.0,'ch is\n RA(J2000) = 22h 25m 02.47s\n Dec(J2000) = +25d 08��� 18.6���\nwith an estimated uncertainty of 5 arcmin radius.'
6,circular_context,NaN,NaN,NaN,NaN,33151.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,2.0,"'ttps://fermi.gsfc.nasa.gov/ssc/data/access/gbm/""\nFSSC: Data � Data Access � GBM - Fermi Gamma-ray Space Telescope<https:/'"
7,circular_context,NaN,NaN,NaN,NaN,33167.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,2.0,' at 5.45 deg from the IC-Cascade-230109A \nbest-fit position.�� In a preliminary analysis of LAT data over 30 days \nbefore'
8,circular_context,NaN,NaN,NaN,NaN,33170.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.0,8.0,"'uni.cz>\n\nJ. Ripa (Masaryk U.), A. Pal (Konkoly Observatory),�� N. Werner (Masaryk 

## Decision 7 — values that are not numbers

**Observed.** Of 32,540 populated `exposure_time_raw` values, 29,968 parse as floats and 2,572 fail across 1,110 distinct forms; the three other quantitative fields have no parse failures. **Why it matters.** Absence and an unparsed textual duration must remain distinguishable. **Decided.** Add a numeric companion only for exposure time while retaining the raw field. **Scope.** The parse decision applies to all 32,540 populated exposure values, with the 30 most frequent failures shown.

In [8]:
parse_rows = []
failure_rows = []
for column in ['exposure_time_raw', 'magnitude_or_limit', 'magnitude_error', 'limit_sigma']:
    values = photometry.loc[photometry[column].map(populated), column].astype(str)
    parsed = pd.to_numeric(values, errors='coerce')
    failed = values[parsed.isna()]
    parse_rows.append({'record_type': 'parse_summary', 'field': column, 'populated_rows': len(values), 'parsed_rows': int(parsed.notna().sum()), 'failed_rows': len(failed), 'distinct_failed_values': int(failed.nunique())})
    if column == 'exposure_time_raw':
        for value, rows in failed.value_counts().head(30).items():
            failure_rows.append({'record_type': 'top_30_exposure_failure', 'field': column, 'raw_value_repr': repr(value), 'rows': int(rows), 'distinct_failed_values_total': int(failed.nunique())})
decision_7_evidence = pd.concat([pd.DataFrame(parse_rows), pd.DataFrame(failure_rows)], ignore_index=True, sort=False)
display(decision_7_evidence)

,record_type,field,populated_rows,parsed_rows,failed_rows,distinct_failed_values,raw_value_repr,rows,distinct_failed_values_total
0,parse_summary,exposure_time_raw,32540.0,29968.0,2572.0,1110.0,NaN,NaN,NaN
1,parse_summary,magnitude_or_limit,37795.0,37795.0,0.0,0.0,NaN,NaN,NaN
2,parse_summary,magnitude_error,3976.0,3976.0,0.0,0.0,NaN,NaN,NaN
3,parse_summary,limit_sigma,1693.0,1693.0,0.0,0.0,NaN,NaN,NaN
4,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'4x90s exposures',71.0,1110.0
5,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'300s',44.0,1110.0
6,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'8x60s',39.0,1110.0
7,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'300 * 6',32.0,1110.0
8,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'3x200 s exposures',30.0,1110.0
9,top_30_exposure_failure,exposure_time_raw,NaN,NaN,NaN,NaN,'600s',28.0,1110.0


## Decision 8 — vocabulary that is not closed

**Observed.** `photometric_band` has 207 distinct states including null: apostrophe variants (`r'`, 171 rows; `r’`, 3), 18 cells carrying a following table value, and two magnitude-with-error strings in the band column. **Why it matters.** These values mix real vocabulary variation with extraction defects, so recoding them here would obscure what the rules emitted. **Decided.** Normalize no free-text value in the corpus build. **Scope.** The complete band inventory and distinct-value counts for `value`, `unit`, `instrument`, and both comment columns are retained as observed.

In [9]:
band_counts = photometry['photometric_band'].value_counts(dropna=False)
band_rows = []
for value, rows in band_counts.items():
    text = value if isinstance(value, str) else None
    if text in {"r'", 'r’'}:
        population = 'same_filter_apostrophe_forms'
    elif isinstance(text, str) and '\t' in text:
        population = 'next_table_column_attached'
    elif isinstance(text, str) and re.fullmatch(r'\d+(?:\.\d+)?±\d+(?:\.\d+)?', text):
        population = 'magnitude_in_band_column'
    else:
        population = 'other_observed_value'
    band_rows.append({'record_type': 'photometric_band_inventory', 'column': 'photometric_band', 'value_repr': repr(value), 'rows': int(rows), 'population': population})
distinct_rows = []
for table, frame, column in [('evidence', evidence, 'value'), ('evidence', evidence, 'unit'), ('photometry', photometry, 'instrument'), ('evidence', evidence, 'comment'), ('photometry', photometry, 'comment')]:
    values = frame.loc[frame[column].map(populated), column]
    distinct_rows.append({'record_type': 'distinct_value_count', 'table': table, 'column': column, 'populated_rows': len(values), 'distinct_values': int(values.nunique())})
decision_8_evidence = pd.concat([pd.DataFrame(band_rows), pd.DataFrame(distinct_rows)], ignore_index=True, sort=False)
display(decision_8_evidence)

,record_type,column,value_repr,rows,population,table,populated_rows,distinct_values
0,photometric_band_inventory,photometric_band,'C',26777.0,other_observed_value,NaN,NaN,NaN
1,photometric_band_inventory,photometric_band,None,2837.0,other_observed_value,NaN,NaN,NaN
2,photometric_band_inventory,photometric_band,'r',1254.0,other_observed_value,NaN,NaN,NaN
3,photometric_band_inventory,photometric_band,'R',593.0,other_observed_value,NaN,NaN,NaN
4,photometric_band_inventory,photometric_band,'i',521.0,other_observed_value,NaN,NaN,NaN
5,photometric_band_inventory,photometric_band,'z',424.0,other_observed_value,NaN,NaN,NaN
6,photometric_band_inventory,photometric_band,'g',388.0,other_observed_value,NaN,NaN,NaN
7,photometric_band_inventory,photometric_band,'P-',343.0,other_observed_value,NaN,NaN,NaN
8,photometric_band_inventory,photometric_band,'P/',336.0,other_observed_value,NaN,NaN,NaN
9,photometric_band_inventory,photometric_band,'P\\\\',283.0,other_observed_value,NaN,NaN,NaN


## Decisions 10 and 11 — time and inherited provenance

**Observed.** All circular publication times are non-null UTC values; 653 circulars were edited later, and their retained annotations belong to edited text. `provenance_inherited` is a serialized JSON list with five token types and nine token-to-column checks, all populated. **Why it matters.** Publication and edit times carry different meanings, while inheritance metadata explains structured values without replacing them. **Decided.** Use `created_on_utc` as public time, retain the edit flag, and preserve serialized provenance. **Scope.** Time handling distinguishes 653 edited circulars; provenance is retained on all 37,795 photometry rows.

In [10]:
created = pd.to_datetime(circulars['created_on_utc'], utc=True, format='mixed')
edited = pd.to_datetime(circulars['edited_on_utc'], utc=True, format='mixed')
edit_mask = circulars['was_edited']
gaps = (edited[edit_mask] - created[edit_mask]).dt.total_seconds() / 3600
edited_ids = set(circulars.loc[edit_mask, 'circular_id'])
time_rows = [['created_on_utc', str(created.dtype), str(created.dt.tz), int(created.isna().sum()), created.min(), created.max(), None, None, None], ['edited_on_utc', str(edited.dtype), str(edited.dt.tz), int(edited.isna().sum()), edited.min(), edited.max(), None, None, None], ['edited_gap_hours', str(gaps.dtype), None, int(gaps.isna().sum()), gaps.min(), gaps.max(), gaps.quantile(.25), gaps.median(), gaps.quantile(.75)], ['edited_circulars', None, None, 0, None, None, None, int(edit_mask.sum()), None], ['annotations_on_edited_circulars', None, None, 0, None, None, None, int(evidence['circular_id'].isin(edited_ids).sum() + photometry['circular_id'].isin(edited_ids).sum()), None]]
decision_10_evidence = pd.DataFrame(time_rows, columns=['field', 'dtype', 'timezone', 'nulls', 'min', 'max', 'p25', 'median_or_count', 'p75'])
display(decision_10_evidence)
parsed_provenance = photometry['provenance_inherited'].map(json.loads)
def token_type(token):
    if token.startswith('secondary_time_col='): return 'secondary_time_col'
    if token.startswith('combined_time_cols='): return 'combined_time_cols'
    if token.startswith('photometric_system='): return 'photometric_system'
    return token.split(':', 1)[0]
token_counts = Counter(token_type(token) for values in parsed_provenance for token in values)
token_fields = {'secondary_time_col': ['obs_time_raw', 'obs_time_type', 'obs_time_reference'], 'combined_time_cols': ['obs_time_raw', 'obs_time_type', 'obs_time_reference'], 'photometric_system': ['photometric_system'], 'system_expected_unknown_for_clear_unfiltered': ['photometric_system'], 'system_from_uvot_convention': ['photometric_system']}
provenance_rows = []
for token, columns in token_fields.items():
    mask = parsed_provenance.map(lambda values: any(token_type(value) == token for value in values))
    for column in columns:
        provenance_rows.append({'storage_type': str(pq.read_schema(corpus_dir / 'photometry_spans.parquet').field('provenance_inherited').type), 'token_type': token, 'token_occurrences': token_counts[token], 'rows_with_token': int(mask.sum()), 'corresponding_column': column, 'populated_rows': int(photometry.loc[mask, column].map(populated).sum()), 'null_or_empty_rows': int((~photometry.loc[mask, column].map(populated)).sum())})
decision_11_evidence = pd.DataFrame(provenance_rows)
display(decision_11_evidence)

,field,dtype,timezone,nulls,min,max,p25,median_or_count,p75
0,created_on_utc,"datetime64[ns, UTC]",UTC,0,2023-01-01 02:26:46+00:00,2026-07-19 15:10:41.186000+00:00,NaN,NaN,NaN
1,edited_on_utc,"datetime64[ns, UTC]",UTC,11359,2024-04-03 18:46:19.777000+00:00,2026-07-15 13:28:22.527000+00:00,NaN,NaN,NaN
2,edited_gap_hours,float64,None,0,0.035096,28791.059736,4.089521,14.989129,48.082382
3,edited_circulars,None,None,0,None,None,NaN,653.000000,NaN
4,annotations_on_edited_circulars,None,None,0,None,None,NaN,5437.000000,NaN


,storage_type,token_type,token_occurrences,rows_with_token,corresponding_column,populated_rows,null_or_empty_rows
0,string,secondary_time_col,29211,29020,obs_time_raw,29020,0
1,string,secondary_time_col,29211,29020,obs_time_type,29020,0
2,string,secondary_time_col,29211,29020,obs_time_reference,29020,0
3,string,combined_time_cols,301,301,obs_time_raw,301,0
4,string,combined_time_cols,301,301,obs_time_type,301,0
5,string,combined_time_cols,301,301,obs_time_reference,301,0
6,string,photometric_system,383,383,photometric_system,383,0
7,string,system_expected_unknown_for_clear_unfiltered,27855,27855,photometric_system,27855,0
8,string,system_from_uvot_convention,1147,1147,photometric_system,1147,0


## Decisions 12 and 13 — the schema and the generation

**Observed.** Runtime model declarations contain 57 vocabulary values across seven fields, 17 of which carry zero rows; 15 extractor versions and 113 rules produced the extracted spans, with six rules firing once. **Why it matters.** Zero-use declared values remain part of the schema, and a generation cannot be identified from tables alone. **Decided.** Preserve declared vocabularies whole and accompany the corpus with a manifest of versions, rules, and hashes. **Scope.** Decision 12 covers all 57 declared values; Decision 13 records one generation containing 101,072 annotations.

In [11]:
from skyportal_corpus.extraction_v2.annotations import EventEvidenceAnnotation
from skyportal_corpus.extraction_v2.photometry_annotations import PhotometricMeasurementAnnotation

def declared_vocabularies(model):
    namespace = vars(inspect.getmodule(model))
    found = {}
    for decorator in model.__pydantic_decorators__.field_validators.values():
        function = getattr(decorator.func, '__func__', decorator.func)
        referenced = [namespace[name] for name in function.__code__.co_names if isinstance(namespace.get(name), frozenset)]
        if len(referenced) == 1:
            for field in decorator.info.fields: found[field] = referenced[0]
    return found
vocabulary_rows = []
for layer, model, frame in [('EVENT_EVIDENCE', EventEvidenceAnnotation, evidence), ('PHOTOMETRIC_MEASUREMENT', PhotometricMeasurementAnnotation, photometry)]:
    for field, values in declared_vocabularies(model).items():
        counts = frame[field].value_counts()
        for value in sorted(values): vocabulary_rows.append([layer, field, value, int(counts.get(value, 0))])
decision_12_evidence = pd.DataFrame(vocabulary_rows, columns=['layer', 'field', 'declared_value', 'rows'])
decision_12_evidence['has_zero_rows'] = decision_12_evidence['rows'].eq(0)
display(decision_12_evidence)
rule_counts = annotations['rule_id'].value_counts()
top_two_share = 100 * rule_counts.head(2).sum() / len(annotations)
generation_rows = [{'record_type': 'summary', 'rules': rule_counts.size, 'top_two_share_pct': round(top_two_share, 3), 'rules_firing_once': int(rule_counts.eq(1).sum()), 'extractors': len(manifest['extractors']), 'annotations': len(annotations)}]
for rule_id, rows in rule_counts[rule_counts.eq(1)].items(): generation_rows.append({'record_type': 'rule_firing_once', 'rule_id': rule_id, 'rows': int(rows)})
for item in manifest['extractors']: generation_rows.append({'record_type': 'extractor_version', **item})
decision_13_evidence = pd.DataFrame(generation_rows)
display(decision_13_evidence)

,layer,field,declared_value,rows,has_zero_rows
0,EVENT_EVIDENCE,label,CLASSIFICATION_INTERPRETATION,2061,False
1,EVENT_EVIDENCE,label,COUNTERPART_ASSOCIATION,3070,False
2,EVENT_EVIDENCE,label,DURATION_GENERAL,931,False
3,EVENT_EVIDENCE,label,EVENT_IDENTITY,27820,False
4,EVENT_EVIDENCE,label,HIGH_ENERGY_PROPERTY,5973,False
5,EVENT_EVIDENCE,label,HOST_CONTEXT,463,False
6,EVENT_EVIDENCE,label,LIGHTCURVE_EVOLUTION,2708,False
7,EVENT_EVIDENCE,label,LOCALIZATION,4912,False
8,EVENT_EVIDENCE,label,NEGATIVE_STATEMENT,978,False
9,EVENT_EVIDENCE,label,REDSHIFT_CONTEXT,65,False


,record_type,rules,top_two_share_pct,rules_firing_once,extractors,annotations,rule_id,rows,extractor_id,extractor_version
0,summary,113.0,46.344,6.0,15.0,101072.0,NaN,NaN,NaN,NaN
1,rule_firing_once,NaN,NaN,NaN,NaN,NaN,duration.t50_explicit,1.0,NaN,NaN
2,rule_firing_once,NaN,NaN,NaN,NaN,NaN,negative_statement.cannot_confirm,1.0,NaN,NaN
3,rule_firing_once,NaN,NaN,NaN,NaN,NaN,classification_interpretation.extended_emission,1.0,NaN,NaN
4,rule_firing_once,NaN,NaN,NaN,NaN,NaN,classification_interpretation.this_is,1.0,NaN,NaN
5,rule_firing_once,NaN,NaN,NaN,NaN,NaN,counterpart_association.suggest,1.0,NaN,NaN
6,rule_firing_once,NaN,NaN,NaN,NaN,NaN,photometry_prose.prose_sigma_depth,1.0,NaN,NaN
7,extractor_version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,event-identity-v1,0.1
8,extractor_version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trigger-time-v1,0.1
9,extractor_version,NaN,NaN,NaN,NaN,NaN,NaN,NaN,localization-v1,0.1


## What this record establishes

Each of the thirteen decisions appears here with the data that motivated
it, measured over the extracted tables and without importing the code
that applies them.

This corpus is built with the extraction rules as they stand. That choice
runs through the document: where a measurement reveals a rule defect —
the uncertainty misassigned across rows carrying several magnitudes, the
classification labels landing in the instrument field, the negation
phrasing no rule covers — the corpus records the output as produced and
the defect is documented. Repairing it here would mean the corpus no
longer reflects what the rules emit.

Two decisions correct nothing and measure a great deal.
`vocabulary_coverage` counts how many of the 57 declared values the rules
emit, and the answer is 40: four of the 17 absent ones are precisely
those annotators had to enter by hand during the validation campaign. And
`needs_review` retains the 11,815 annotations where the rule itself
states what it could not resolve, each with its comment.

The corpus is accompanied by a manifest carrying the extractor versions,
the rule inventory and the hash of each table. The project's other two
corpora do not carry one, because they do not need it: they derive from
frozen captures rather than from code that will change.